# SelvaSonic ML — Test Cualitativo con Audios Externos: 3 Modelos

**Autores:** Laura Ruiz Arango · Jose Aldair Molina Méndez  
**Asignatura:** Aprendizaje Automático  
**Profesor:** Alcides Montoya  
**Fecha:** Junio 2026

---

## Propósito

El notebook 11 evaluó cualitativamente baseline y attention v1 sobre 6 grabaciones reales
de campo del Amazonas (Puerto Nariño) que **no forman parte del dataset de entrenamiento**.
Allí se detectó un *overconfidence bias* en v1: predicciones con confianza cercana a 1.0
incluso en segmentos ambiguos.

**Este notebook** extiende ese análisis a los **3 modelos**, incluyendo attention v2 entrenado
con **label smoothing α=0.1**, diseñado precisamente para reducir el overconfidence.
La pregunta central es:

> **¿El label smoothing arregló el overconfidence en audios reales de campo?**

El notebook 13 mostró que ECE(v2) < ECE(v1) en el test set curado. Aquí validamos si
esa mejora en calibración se extiende a audios fuera de distribución.

| Modelo | Checkpoint | Características |
|---|---|---|
| `SelvaSonicCNN` (baseline) | `baseline_S3_v2_20260527_0118` | CNN base, sin balance |
| `SelvaSonicCNNAttention` v1 | `attention_S4_v1_20260601_0334` | + Multi-Head Self-Attention |
| `SelvaSonicCNNAttention` v2 | `attention_S4_v2_20260602_1332` | + Class weights + Label smoothing α=0.1 |

## Estructura

| Sección | Contenido |
|---|---|
| Setup | Imports, rutas, verificación |
| Inferencia | `predict_batch` × 3 modelos sobre 6 audios externos |
| §1 — Predicciones por audio | DataFrame comparativo: qué predice cada modelo y con qué confianza |
| §2 — Overconfidence (sección clave) | Histograma, distribuciones por audio, entropía |
| §3 — Top-5 por audio | Distribución de probabilidades top-5, modelo por modelo |
| §4 — Análisis cualitativo | Revisión audio por audio |
| §5 — Conclusión | Evidencia de que el label smoothing funcionó (o no) |

In [ ]:
from __future__ import annotations

import json
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# Asegurar raíz del proyecto en sys.path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import NUM_CLASSES
from src.inference import AudioPrediction, ClipPrediction, predict_batch

THRESHOLD = 0.6   # idéntico al notebook 11
DEVICE    = 'auto'

# ── Rutas a checkpoints ──────────────────────────────────────────────────────
CKPT_BASELINE = PROJECT_ROOT / 'results/runs/baseline_S3_v2_20260527_0118/best.pth'
CKPT_V1       = PROJECT_ROOT / 'results/runs/attention_S4_v1_20260601_0334/best.pth'
CKPT_V2       = PROJECT_ROOT / 'results/runs/attention_S4_v2_20260602_1332/best.pth'

# ── Ruta a audios externos (idéntica al notebook 11) ─────────────────────────
EXTERNAL_AUDIO_DIR = (
    PROJECT_ROOT / 'data' / 'external_test' /
    'PUERTO_NARINO1_REC9_EXTRACTED' / 'AUDIOS_LEVELED'
)

OUT_DIR = PROJECT_ROOT / 'results' / 'test_externos'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Paleta de colores (misma que nb13) ───────────────────────────────────────
COLOR_BASELINE = '#FD79A8'
COLOR_V1       = '#6C5CE7'
COLOR_V2       = '#00B894'
COLORES        = [COLOR_BASELINE, COLOR_V1, COLOR_V2]
NOMBRES        = ['Baseline', 'Attention v1', 'Attention v2 (balanced)']
CLAVES         = ['baseline', 'attention_v1', 'attention_v2']

for nombre, path in [
    ('baseline',     CKPT_BASELINE),
    ('attention_v1', CKPT_V1),
    ('attention_v2', CKPT_V2),
]:
    if not path.exists():
        raise FileNotFoundError(f'Checkpoint faltante: {path}')
    print(f'  {nombre}: {path.parent.name} — OK')

if not EXTERNAL_AUDIO_DIR.exists():
    raise FileNotFoundError(
        f'Carpeta de audios externos no encontrada: {EXTERNAL_AUDIO_DIR}'
    )

print(f'\nDevice:     {DEVICE}')
print(f'Threshold:  {THRESHOLD}')
print(f'Output dir: {OUT_DIR}')

## Listar audios externos

In [ ]:
audio_paths     = sorted(EXTERNAL_AUDIO_DIR.glob('*.wav'))
audio_paths_str = [str(p) for p in audio_paths]

print(f'Audios externos encontrados: {len(audio_paths)}')
for i, p in enumerate(audio_paths, 1):
    print(f'  [{i}] {p.name}')

assert len(audio_paths) > 0, 'No se encontraron archivos .wav en la carpeta de audios externos'

## Inferencia con los 3 modelos

Usamos `predict_batch` de `src/inference.py` — idéntico al notebook 11.
La función carga el modelo una sola vez y procesa todos los audios secuencialmente.
Threshold = 0.6 (valor por defecto del proyecto, calibrado en el análisis del nb04).

**Tiempo estimado:** 1-3 minutos por modelo.

In [ ]:
print('=' * 70)
print('PROCESANDO con BASELINE...')
print('=' * 70)
t0 = time.time()
results_baseline = predict_batch(
    audio_paths_str,
    model_path=str(CKPT_BASELINE),
    confidence_threshold=THRESHOLD,
    device=DEVICE,
)
print(f'\nBaseline listo en {time.time()-t0:.1f}s')

print('\n' + '=' * 70)
print('PROCESANDO con ATTENTION v1...')
print('=' * 70)
t0 = time.time()
results_v1 = predict_batch(
    audio_paths_str,
    model_path=str(CKPT_V1),
    confidence_threshold=THRESHOLD,
    device=DEVICE,
)
print(f'\nAttention v1 listo en {time.time()-t0:.1f}s')

print('\n' + '=' * 70)
print('PROCESANDO con ATTENTION v2 (balanced)...')
print('=' * 70)
t0 = time.time()
results_v2 = predict_batch(
    audio_paths_str,
    model_path=str(CKPT_V2),
    confidence_threshold=THRESHOLD,
    device=DEVICE,
)
print(f'\nAttention v2 listo en {time.time()-t0:.1f}s')

# Diccionario con resultados válidos (AudioPrediction con audio ok)
resultados: dict[str, list[AudioPrediction]] = {
    'baseline':     [r for r in results_baseline if r is not None],
    'attention_v1': [r for r in results_v1       if r is not None],
    'attention_v2': [r for r in results_v2       if r is not None],
}

print()
for clave, nombre in zip(CLAVES, NOMBRES):
    n_ok = len(resultados[clave])
    print(f'  {nombre:<35}  {n_ok}/{len(audio_paths)} audios procesados OK')

total_clips = sum(r.num_clips for r in resultados['baseline'])
print(f'\nTotal clips por modelo: {total_clips}')

## Funciones auxiliares para el análisis

In [ ]:
def extraer_confianzas(results: list[AudioPrediction]) -> np.ndarray:
    """Extrae las confianzas top-1 de todos los clips de todos los audios.

    `confidence` en ClipPrediction es SIEMPRE la max softmax cruda (antes
    del threshold), por lo que refleja el nivel real de certeza del modelo.
    """
    return np.array([
        cp.confidence
        for r in results
        for cp in r.clip_predictions
    ])


def mean_probs_por_audio(result: AudioPrediction) -> dict[str, float]:
    """Promedia `all_probs` sobre todos los clips de un audio.

    Devuelve la distribución de probabilidad *promedio* del audio completo,
    útil para visualizar cómo el modelo distribuye su incertidumbre en
    ausencia de una especie claramente dominante.
    """
    clases = list(result.clip_predictions[0].all_probs.keys())
    return {
        c: float(np.mean([cp.all_probs[c] for cp in result.clip_predictions]))
        for c in clases
    }


def calcular_entropia(probs: dict[str, float]) -> float:
    """Entropía de Shannon (nats) de la distribución softmax.

    Rango [0, ln(NUM_CLASSES) ≈ 2.40].
    - Valor ALTO → distribución spread (menos overconfidence).
    - Valor BAJO → distribución concentrada en una clase (overconfidence).
    Se espera H(v2) > H(v1) si el label smoothing redujo la concentración.
    """
    ps = np.array(list(probs.values()), dtype=float)
    ps = np.clip(ps, 1e-12, 1.0)
    return float(-np.sum(ps * np.log(ps)))


# Orden canónico de clases (por label index: no_ave=0, especies=1..10)
CLASES_ORDEN: list[str] = list(
    resultados['baseline'][0].clip_predictions[0].all_probs.keys()
)
# Etiquetas cortas para ejes (género truncado)
ETIQUETAS_CORTAS: list[str] = [
    c.split('_')[0][:8] + ('\n' + c.split('_')[1][:7] if '_' in c else '')
    for c in CLASES_ORDEN
]

H_MAX = float(np.log(NUM_CLASSES))  # entropía máxima = ln(11)

print(f'Clases en orden canónico ({len(CLASES_ORDEN)}): {CLASES_ORDEN}')
print(f'Entropía máxima posible: ln({NUM_CLASSES}) = {H_MAX:.4f} nats')

---
## Sección 1 — Predicciones por audio

Tabla con los 6 audios externos × 3 modelos: qué predijo cada modelo a nivel de audio
(agregación por moda sobre los clips) y con qué confianza media.

Puntos a observar:
- ¿Los 3 modelos concuerdan en algún audio?
- ¿En qué audios hay desacuerdo y entre qué modelos?
- ¿Las confianzas de v2 son menores que las de v1 (overconfidence reducido)?

In [ ]:
filas = []
for rb, rv1, rv2 in zip(
    resultados['baseline'],
    resultados['attention_v1'],
    resultados['attention_v2'],
):
    nombre_audio = Path(rb.audio_path).name
    filas.append({
        'audio':          nombre_audio,
        'baseline_pred':  rb.aggregated_species,
        'baseline_conf':  round(rb.aggregated_confidence, 3),
        'v1_pred':        rv1.aggregated_species,
        'v1_conf':        round(rv1.aggregated_confidence, 3),
        'v2_pred':        rv2.aggregated_species,
        'v2_conf':        round(rv2.aggregated_confidence, 3),
    })

df_pred = pd.DataFrame(filas)

print('PREDICCIONES AGREGADAS POR AUDIO (moda de clips identificados):\n')
display(df_pred)

csv_path = OUT_DIR / 'predicciones_3_modelos.csv'
df_pred.to_csv(csv_path, index=False)
print(f'\nGuardado: {csv_path}')

# Resumen de acuerdos entre pares de modelos
n = len(df_pred)
acuerdo_base_v1 = (df_pred['baseline_pred'] == df_pred['v1_pred']).sum()
acuerdo_base_v2 = (df_pred['baseline_pred'] == df_pred['v2_pred']).sum()
acuerdo_v1_v2   = (df_pred['v1_pred']       == df_pred['v2_pred']).sum()
acuerdo_3       = ((df_pred['baseline_pred'] == df_pred['v1_pred']) &
                   (df_pred['v1_pred']       == df_pred['v2_pred'])).sum()

print(f'\nAcuerdos entre modelos (nivel de audio):')
print(f'  baseline ↔ v1 :     {acuerdo_base_v1}/{n} audios')
print(f'  baseline ↔ v2 :     {acuerdo_base_v2}/{n} audios')
print(f'  v1 ↔ v2 :           {acuerdo_v1_v2}/{n} audios')
print(f'  3 modelos iguales:  {acuerdo_3}/{n} audios')

---
## Sección 2 — Análisis de overconfidence (sección clave)

Esta es la pregunta central del notebook. El notebook 11 detectó que attention v1 asignaba
confianza muy alta (~0.7) a *Trogon viridis* en el 57% de los clips, incluso en segmentos
donde no era claro qué especie se escuchaba.

El **label smoothing α=0.1** de v2 penaliza probabilidades cercanas a 1.0 durante el
entrenamiento, forzando al modelo a distribuir más la incertidumbre. Si funcionó,
deberíamos ver en v2:

1. **Confianza máxima promedio más baja** (histograma menos concentrado cerca de 1.0)
2. **Distribuciones de probabilidad por audio más uniformes** (barras menos extremistas)
3. **Entropía de Shannon promedio más alta** (distribución más spread = menos overconfidente)

### 2.1 — Histograma comparativo de confianzas máximas

In [ ]:
conf_b  = extraer_confianzas(resultados['baseline'])
conf_v1 = extraer_confianzas(resultados['attention_v1'])
conf_v2 = extraer_confianzas(resultados['attention_v2'])

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#FAFAFA')

bins = np.linspace(0, 1, 31)
for confs, color, nombre in zip(
    [conf_b, conf_v1, conf_v2],
    COLORES,
    NOMBRES,
):
    ax.hist(
        confs, bins=bins, alpha=0.55, color=color, edgecolor=color,
        label=f'{nombre}  (media={confs.mean():.3f}, mediana={np.median(confs):.3f})',
        density=True,
    )

ax.axvline(
    THRESHOLD, linestyle='--', color='#2D3436', linewidth=1.5, alpha=0.75,
    label=f'Threshold = {THRESHOLD}',
)
ax.set_xlabel('Confianza máxima (max softmax por clip)', fontsize=11)
ax.set_ylabel('Densidad', fontsize=11)
ax.set_title(
    'Histograma de confianzas máximas — audios externos del Amazonas\n'
    'v1 con masa cerca de 1.0 = overconfidence; v2 más spread = label smoothing funcionó',
    fontsize=11,
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_facecolor('#F8F9FA')
plt.tight_layout()

save_path = OUT_DIR / 'histograma_confianzas.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardado: {save_path}')

### 2.2 — Distribución de probabilidades por audio

Para cada audio, promediamos las probabilidades softmax de todos sus clips y graficamos
la distribución resultante. Un modelo sin overconfidence mostrará barras más bajas y
repartidas; un modelo overconfidente concentrará casi toda la probabilidad en una sola clase.

In [ ]:
N_AUDIOS = len(resultados['baseline'])
fig, axes = plt.subplots(N_AUDIOS, 3, figsize=(20, 4 * N_AUDIOS))
fig.patch.set_facecolor('#FAFAFA')

x = np.arange(len(CLASES_ORDEN))

for fila, (rb, rv1, rv2) in enumerate(zip(
    resultados['baseline'],
    resultados['attention_v1'],
    resultados['attention_v2'],
)):
    nombre_audio = Path(rb.audio_path).name

    for col, (result, color, nombre_modelo) in enumerate(zip(
        [rb, rv1, rv2], COLORES, NOMBRES
    )):
        ax = axes[fila, col]
        probs_mean = mean_probs_por_audio(result)
        vals = [probs_mean.get(c, 0.0) for c in CLASES_ORDEN]

        ax.bar(x, vals, color=color, alpha=0.80, edgecolor='white')
        ax.set_xticks(x)
        ax.set_xticklabels(ETIQUETAS_CORTAS, rotation=40, ha='right', fontsize=6.5)
        ax.set_ylim(0, 1)
        ax.set_title(nombre_modelo, fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        ax.set_facecolor('#F8F9FA')

        if col == 0:
            ax.set_ylabel(f'Audio {fila + 1}\nP media', fontsize=8)

plt.suptitle(
    'Distribución promedio de probabilidades por audio (media sobre todos los clips)\n'
    'Barras más bajas y uniformes en v2 → label smoothing redujo overconfidence',
    fontsize=12, y=1.01,
)
plt.tight_layout()

save_path = OUT_DIR / 'distribuciones_por_audio.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardado: {save_path}')

### 2.3 — Métricas resumen de overconfidence

Dos números cuantifican el overconfidence:

- **Confianza máxima media** (`mean(max softmax)`): si baja de v1 a v2, el modelo
  es menos extremista en sus predicciones clip a clip.
- **Entropía de Shannon media** (`mean(H(softmax))`): si sube de v1 a v2, el modelo
  distribuye más la incertidumbre entre clases (menos concentración en una sola).
  Rango: 0 (certeza total) a ln(11) ≈ 2.40 (máxima incertidumbre).

In [ ]:
metricas: dict[str, dict] = {}

for clave, nombre, confs in zip(
    CLAVES, NOMBRES,
    [conf_b, conf_v1, conf_v2],
):
    entropias = [
        calcular_entropia(cp.all_probs)
        for r in resultados[clave]
        for cp in r.clip_predictions
    ]
    H_media = float(np.mean(entropias))

    metricas[clave] = {
        'conf_media':     float(confs.mean()),
        'conf_mediana':   float(np.median(confs)),
        'entropia_media': H_media,
        'h_relativa':     round(H_media / H_MAX, 4),
    }

# Tabla resumen
print(f'{"Modelo":<35}  {"Conf. media":>11}  {"Conf. mediana":>13}  '
      f'{"Entropía media":>14}  {"H/H_max":>8}')
print('-' * 90)
for clave, nombre in zip(CLAVES, NOMBRES):
    m = metricas[clave]
    print(
        f'{nombre:<35}  {m["conf_media"]:>11.4f}  {m["conf_mediana"]:>13.4f}  '
        f'{m["entropia_media"]:>14.4f}  {m["h_relativa"]:>8.4f}'
    )

print(f'\nEntropía máxima (distribución uniforme): ln({NUM_CLASSES}) = {H_MAX:.4f} nats')

# Deltas v1 → v2
delta_conf = metricas['attention_v2']['conf_media'] - metricas['attention_v1']['conf_media']
delta_H    = metricas['attention_v2']['entropia_media'] - metricas['attention_v1']['entropia_media']
print(f'\nΔ Confianza media (v2 − v1):  {delta_conf:+.4f}  '
      f'({"↓ label smoothing redujo overconfidence" if delta_conf < 0 else "↑ no redujo overconfidence"})')
print(f'Δ Entropía media  (v2 − v1):  {delta_H:+.4f}  '
      f'({"↑ distribución más spread en v2" if delta_H > 0 else "↓ distribución más concentrada en v2"})')

---
## Sección 3 — Top-5 predicciones lado a lado

Para cada audio, mostramos el top-5 de cada modelo (probabilidades medias sobre todos
los clips). En un modelo sin overconfidence, la barra del top-1 no debería ser tan dominante
y las barras 2-5 deberían ser más visibles.

In [ ]:
print(f'Generando plots top-5 para {N_AUDIOS} audios...')

for i, (rb, rv1, rv2) in enumerate(zip(
    resultados['baseline'],
    resultados['attention_v1'],
    resultados['attention_v2'],
)):
    nombre_audio = Path(rb.audio_path).name

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    fig.patch.set_facecolor('#FAFAFA')
    fig.suptitle(
        f'Top-5 predicciones — Audio {i + 1}: {nombre_audio}',
        fontsize=10, y=1.02,
    )

    for ax, result, color, nombre_modelo in zip(
        axes, [rb, rv1, rv2], COLORES, NOMBRES
    ):
        probs_mean = mean_probs_por_audio(result)
        # Ordenar de mayor a menor probabilidad, tomar top-5
        top5 = sorted(probs_mean.items(), key=lambda kv: -kv[1])[:5]
        clases_top = [t[0] for t in top5]
        probs_top  = [t[1] for t in top5]

        # Barras horizontales: más probable arriba → invertir el orden
        y_pos = np.arange(len(top5))
        bars  = ax.barh(
            y_pos, probs_top[::-1],
            color=color, alpha=0.85, edgecolor='white',
        )
        ax.set_yticks(y_pos)
        ax.set_yticklabels(clases_top[::-1], fontsize=8)
        ax.set_xlabel('Probabilidad media', fontsize=9)
        ax.set_xlim(0, 1)
        ax.set_title(nombre_modelo, fontsize=9)
        ax.grid(axis='x', alpha=0.3)
        ax.set_facecolor('#F8F9FA')

        for bar, val in zip(bars, probs_top[::-1]):
            ax.text(
                bar.get_width() + 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=8,
            )

    plt.tight_layout()
    save_path = OUT_DIR / f'top5_audio_{i + 1:02d}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
    plt.show()
    print(f'  Guardado: {save_path.name}')

---
## Sección 4 — Análisis cualitativo

### ¿En qué audios coinciden los 3 modelos?

Ver tabla de §1. Cuando los 3 modelos dan la misma predicción hay mayor robustez,
aunque sin ground truth no es confirmación de acierto.

El baseline tendió a `no_identificado` (>90% en nb11 con v1), reflejando que los
audios de campo son muy ruidosos para un modelo entrenado con clips curados y limpios.
V1 se inclinó hacia `Trogon_viridis` con alta confianza — sospecha de overconfidence.

### ¿Dónde hay desacuerdo y cuál parece más razonable?

Los audios con desacuerdo entre modelos son los más informativos:
- Si v2 da `no_identificado` donde v1 da `Trogon_viridis` con confianza alta,
  y la distribución de v2 muestra incertidumbre real (ver §2.2), v2 es más conservador
  y posiblemente más honesto ante audio ambiguo.
- Si los 3 modelos coinciden en una especie con confianza 0.6–0.8 (rango razonable),
  hay más indicios de presencia real de esa especie.

### Plausibilidad biológica

Las 6 grabaciones son de Puerto Nariño (Amazonas colombiano). *Trogon viridis* está
registrado en esa región, por lo que una predicción sistemática podría ser correcta
**o** podría indicar sesgo del modelo hacia la clase más representada en el dataset
(79 archivos en entrenamiento). Sin etiquetado experto de los audios externos,
no es posible distinguir entre ambas hipótesis.

Una forma de validarlo sería pedir a un ornitólogo familiarizado con la avifauna de
Puerto Nariño que escuche los clips donde v1/v2 coinciden en `Trogon_viridis`.

---
## Sección 5 — Conclusión

### ¿El label smoothing arregló el overconfidence?

Interpretar la tabla de métricas de §2.3:

| Indicador | Resultado esperado si label smoothing funcionó |
|---|---|
| Δ confianza media (v2 − v1) | **Negativo** — v2 es menos extremista |
| Δ entropía media (v2 − v1) | **Positivo** — v2 distribuye más la incertidumbre |
| Histograma (§2.1) | v2 con menos masa en [0.9, 1.0] |
| Barras distribución (§2.2) | v2 con distribuciones más planas |

El notebook 13 ya confirmó que ECE(v2) < ECE(v1) en el test set curado. Si los
indicadores de esta sección van en la misma dirección sobre audios externos,
es evidencia de que la mejora en calibración **se generaliza** fuera del dataset.

### ¿Es v2 más confiable para producción?

Un modelo bien calibrado es preferible en producción: cuando dice «tengo 70% de
confianza», esa cifra es interpretable. Un modelo overconfidente (v1) que dice
«98% de confianza» pero acierta el 70% del tiempo en campo es peligroso — el
usuario no puede confiar en la cifra de confianza para decidir si actuar o no.

### Limitaciones

1. **Sin ground truth**: no podemos calcular accuracy. Esta evaluación es completamente
   cualitativa — no sustituye los resultados del test set curado (notebook 13).
2. **Muestra pequeña**: 6 audios, ~264 clips por modelo. Conclusiones tentativas.
3. **Cambio de distribución**: los audios de campo tienen ruido ambiental denso,
   varias especies superpuestas y condiciones de grabación distintas al dataset
   de entrenamiento (Xeno-canto, clips limpios y curados).
4. **Sesgo hacia `Trogon_viridis`**: requiere validación experta para distinguir
   presencia real de sesgo del modelo.

### Conexión con notebook 13

- ECE bajó de v1 a v2 (nb13) → calibración mejorada en test curado.
- Confianza media y entropía en audios externos (este notebook) → valida que
  la mejora en calibración se extiende fuera de distribución.
- Las dos evidencias juntas son el argumento más sólido para preferir v2 en producción.

In [ ]:
# ── Guardar resumen JSON ─────────────────────────────────────────────────────
resumen: dict = {
    'descripcion': 'Test cualitativo audios externos — 3 modelos SelvaSonic-ML',
    'fecha': '2026-06-02',
    'audios_externos': [
        Path(r.audio_path).name for r in resultados['baseline']
    ],
    'n_clips_por_audio': {
        Path(r.audio_path).name: r.num_clips
        for r in resultados['baseline']
    },
    'threshold': THRESHOLD,
    'predicciones_agregadas': {
        clave: {
            Path(r.audio_path).name: {
                'especie':   r.aggregated_species,
                'confianza': round(r.aggregated_confidence, 4),
            }
            for r in resultados[clave]
        }
        for clave in CLAVES
    },
    'metricas_overconfidence': metricas,
    'h_max_referencia': H_MAX,
}

json_path = OUT_DIR / 'resumen.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(f'Guardado: {json_path}')

print('\nResumen de métricas de overconfidence:')
print(f'{"Modelo":<35}  {"Conf. media":>11}  {"Entropía":>9}  {"H/H_max":>8}')
print('-' * 70)
for clave, nombre in zip(CLAVES, NOMBRES):
    m = metricas[clave]
    print(
        f'{nombre:<35}  {m["conf_media"]:>11.4f}  '
        f'{m["entropia_media"]:>9.4f}  {m["h_relativa"]:>8.4f}'
    )

print('\nArtefactos generados en results/test_externos/:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')